# 2장 2강: Likelihood와 Maximum Likelihood — 실습문제

## 실습 목표

- 조건을 만족하는 고객만 골라 구매 비율을 구합니다.
- 실제 구매 여부에 맞는 확률을 선택하고, 그 확률들을 곱해 우도(Likelihood)를 구합니다.
- 몇 가지 구매 확률 후보를 비교하여 최대우도추정(MLE)의 뜻을 익힙니다.
- 로그우도와 음의 로그우도(NLL)를 단계별로 계산하고 합계와 평균을 구분합니다.

## 실습 환경 / 데이터

Python과 NumPy를 사용합니다. 외부 데이터 파일은 필요하지 않습니다. 모든 데이터와 수학적 계산 방법은 제공된 「2장 2강: Likelihood와 Maximum Likelihood」의 본문과 연습문제에서 가져왔습니다. **0은 미구매, 1은 구매**입니다. 이 자료에서는 확률이 주어졌을 때의 계산을 연습합니다.

## 실습 구성과 교안 대응

| 구분 | 실습 내용 | 교안 근거 |
|---|---|---|
| 필수 1 / 문제 1-1 | 전체·장바구니 추가 여부별 구매 비율 | 1절 본문: 고객 10명의 조건부 확률 예제 |
| 필수 1 / 문제 1-2 | 공통 구매 확률 0.6과 0.8의 우도 | 2절 본문: 관측값 [1,0,1,1,0] 비교 |
| 필수 2 / 문제 2-1 | 네 후보의 우도와 구매 비율로 MLE 확인 | 3절 본문: 후보 0.2, 0.4, 0.6, 0.8 |
| 필수 2 / 문제 2-2 | 고객별 NLL, 전체 로그우도, 합계·평균 NLL | 교안 후반부: y=[1,0,1], p=[0.8,0.3,0.7] 예제 |
| 과제 / 문제 3-1 | 세 후보의 우도와 MLE 복습 | 연습문제 3: y=[1,1,0,1], 후보 0.5, 0.75, 0.9 |
| 과제 / 문제 3-2 | 고객별 확률이 다른 두 모델의 우도 | 연습문제 2: 두 모델의 고객별 확률 비교 |

**교안 연습문제 전체를 그대로 옮긴 자료는 아닙니다.** 본문 예제와 연습문제의 일부를 필수 실습 4문제와 복습 과제 2문제로 재구성했습니다. 과제도 앞에서 연습한 식에 교안의 숫자를 넣어 계산하는 기초 수준입니다.

## 진행 안내

1. NumPy를 불러오는 셀부터 위에서 아래로 실행합니다.
2. 각 문제의 표와 기호 설명을 읽고 계산하세요. 함수가 제시된 경우 그대로 사용해도 됩니다.
3. 손으로 로그값을 구하거나 공식을 증명할 필요는 없습니다. 로그 계산은 교안의 `np.log()`를 사용합니다.
4. 계산 중간에는 반올림하지 않습니다. 마지막 출력에만 `round()` 또는 `np.round()`를 사용합니다.
5. 각 문제의 해석 질문 4개에 계산값과 짧은 설명을 적습니다.

## 먼저 읽는 용어와 기호

| 용어·기호 | 쉬운 뜻 |
|---|---|
| 관측 데이터 D / 실제값 y | 이미 확인한 실제 결과들 / 한 고객의 구매 여부. 코드에서는 y에 여러 결과를 순서대로 넣음 |
| 확률 p | 구매할 가능성을 나타내는 수. 0.6은 60% |
| 1−p | 구매하지 않을 확률. 구매와 미구매의 확률을 더하면 1 |
| 베르누이 분포 | 결과가 0과 1 두 가지인 경우의 확률을 나타내는 방법 |
| 조건부 확률 | 특정 조건을 만족하는 경우만 모아서 보는 확률 |
| P(구매 ∣ 장바구니 추가) | 장바구니에 추가했다는 조건에서의 구매 확률. 세로줄 뒤가 조건 |
| 파라미터 | 모델의 확률을 결정하는 값. 공통 구매 확률 예제에서는 p |
| 독립 가정 | 여기서는 p가 정해졌을 때 한 고객의 구매 여부가 다른 고객의 구매 여부에 영향을 주지 않는다고 가정함 |
| 우도 L(p) / Likelihood | 실제로 관측한 데이터를 고정하고, p에 따라 그 데이터에 부여되는 확률을 비교하는 값 |
| 최대우도추정 / MLE | 관측 데이터의 우도가 가장 커지는 파라미터를 찾는 것 |
| p̂ / ‘피 햇’ | 데이터를 보고 추정한 p. 위에 붙은 작은 모자는 추정값이라는 표시 |
| ln / 자연로그 | 밑이 자연상수 e인 로그. 코드의 `np.log()`로 계산 |
| 로그우도 / Log-Likelihood | 우도에 자연로그를 적용한 값. 관측 확률 각각의 로그를 더해 계산할 수 있음 |
| 음의 로그우도 / NLL | 로그우도의 부호를 바꾼 값. 이 자료에서는 작을수록 좋다고 보는 손실값 |
| Total / Mean | 합계 / 평균. 평균은 합계를 데이터 개수로 나눈 값 |
| ≈ / Σ / Π | 대략 같음 / 여러 값을 더함 / 여러 값을 곱함 |
| p³ / p ** 3 | 둘 다 p를 세 번 곱한다는 뜻 |
| 인덱스 | 배열의 위치 번호. 첫 번째 위치는 0, 두 번째는 1 |

**가장 먼저 구분할 것:** p는 항상 ‘구매 확률’입니다. 실제로 구매했다면 p를, 구매하지 않았다면 1−p를 사용합니다. 실제 결과에 해당하는 확률을 고른 다음 우도와 NLL을 계산합니다.

## 실습 준비

아래 셀을 먼저 실행하세요. NumPy는 숫자 배열을 계산하는 도구이며, `np`는 NumPy를 짧게 부르는 이름입니다.

In [ ]:
import numpy as np

---

## 과제. 앞에서 배운 기본 계산 복습

### 문제 3-1. 새로운 데이터의 우도와 MLE 복습하기

#### 문제 설명

교안 연습문제 3의 실제 구매 결과 `y = [1, 1, 0, 1]`과 구매 확률 후보 `[0.5, 0.75, 0.9]`를 사용합니다. 모든 고객의 구매 확률이 공통 p이고, p가 주어졌을 때 구매 여부는 서로 독립이라고 가정합니다.

문제 2-1의 우도 함수와 후보 비교 코드를 그대로 사용하고, 데이터와 후보만 바꾸어 계산하세요.

#### 요구사항

1. 구매 수를 `np.sum(y)`, 미구매 수를 `len(y) - np.sum(y)`로 구해 출력하세요.
2. 다음 우도 함수로 세 후보의 우도를 구하세요. 소수점 아래 여덟 자리까지 표시합니다.
3. `np.argmax()`로 가장 큰 우도의 위치를 찾고 해당 p를 출력하세요. `np.mean(y)`로 MLE도 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

```python
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

values = np.array([likelihood(p, y) for p in candidates])
best_index = np.argmax(values)
best_p = candidates[best_index]
```

문제 2-1과 같은 순서입니다. 구매가 몇 번, 미구매가 몇 번인지 먼저 세면 L(p)의 거듭제곱 식도 쓸 수 있습니다.

#### 해석 질문

**Q1.** 구매·미구매 수와 이 데이터의 우도 식 L(p)는 무엇인가요?

**Q2.** 세 후보의 우도는 각각 얼마이며, 가장 좋은 후보는 무엇인가요?

**Q3.** 최대 우도의 위치 번호와 관측 평균으로 구한 MLE는 각각 무엇인가요?

**Q4.** p=0.9가 p=0.75보다 큰데도 가장 좋은 후보가 아닌 이유는 무엇인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-1. 새로운 데이터의 우도와 MLE 복습하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

In [ ]:
y= np.array([1, 1, 0, 1])
candidates= np.array([0.5, 0.75, 0.9])
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

values = np.array([likelihood(p, y) for p in candidates])
best_index = np.argmax(values)
best_p = candidates[best_index]

In [ ]:
print("np.sum(y): \n", np.sum(y))
print("\nlen(y) - np.sum(y): \n", len(y) - np.sum(y))

np.sum(y): 
 3

len(y) - np.sum(y): 
 1


In [ ]:
values = np.array([likelihood(p, y) for p in candidates])
print("values: \n", values)

values: 
 [0.0625     0.10546875 0.0729    ]


In [ ]:
best_index = np.argmax(values)
best_p = candidates[best_index]
print("best_index: \n", best_index)
print("\nbest_p: \n", best_p)
print("\nnp.mean(y): \n", np.mean(y))

best_index: 
 1

best_p: 
 0.75

np.mean(y): 
 0.75


#### 답변 작성란

**Q1.** 구매·미구매 수와 이 데이터의 우도 식 L(p)는 무엇인가요?

- 답변: 구매 3건, 미구매 1건
- 우도 식: L(p) = p × p × (1−p) × p = p³(1−p)
```python
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)
```

**Q2.** 세 후보의 우도는 각각 얼마이며, 가장 좋은 후보는 무엇인가요?

- 답변: 
```python
values: 
 [0.0625     0.10546875 0.0729    ]
```
- 우도는 p=0.5일 때 0.06250000, p=0.75일 때 0.10546875, p=0.9일 때 0.07290000입니다. 가장 좋은 후보는 우도가 가장 큰 p=0.75입니다.

**Q3.** 최대 우도의 위치 번호와 관측 평균으로 구한 MLE는 각각 무엇인가요?

- 답변: 최대 우도의 위치 번호는 인덱스 1에 해당하는 위치이다.
- 관측 평균으로 구한 MLE는 0.75이다.

**Q4.** p=0.9가 p=0.75보다 큰데도 가장 좋은 후보가 아닌 이유는 무엇인가요?

- 우도는 구매한 고객과 미구매 고객의 확률을 모두 곱한 값입니다.  
p=0.9로 두면 구매 고객 3명에게는 0.9³ = 0.729로 높은 확률을 주지만, 미구매 고객 1명에게는 1−0.9 = 0.1밖에 주지 못합니다.  
곱하면 0.0729가 되어 p=0.75의 0.10546875보다 작아집니다.  
즉 p가 크다고 항상 좋은 것이 아니라, 구매와 미구매를 함께 가장 잘 설명하는 값이 좋습니다.  
그 값이 구매 비율 3/4 = 0.75입니다.

### 문제 3-2. 고객별 확률이 다른 두 모델의 우도 비교하기

#### 문제 설명

교안 연습문제 2의 두 모델 예제입니다. 표의 구매 확률은 이미 계산된 값입니다. 각 모델의 입력 특성과 파라미터가 주어졌을 때 고객들의 구매 여부는 서로 조건부 독립이라고 가정합니다.

| 고객 | 실제 구매 여부 y | 모델 A의 구매 확률 | 모델 B의 구매 확률 |
|---|---:|---:|---:|
| 첫 번째 고객 | 1 | 0.8 | 0.6 |
| 두 번째 고객 | 0 | 0.4 | 0.2 |
| 세 번째 고객 | 1 | 0.6 | 0.5 |

문제 1-2와 같이 실제 결과에 해당하는 확률을 고르고 곱하세요. 이번에는 공통 p 하나 대신 고객별 p 배열이 주어집니다. 배열의 같은 위치가 같은 고객입니다.

#### 요구사항

1. `y = np.array([1,0,1])`, `p_a = np.array([0.8,0.4,0.6])`, `p_b = np.array([0.6,0.2,0.5])`를 만드세요.
2. `(p ** y) * ((1-p) ** (1-y))`를 두 모델에 각각 적용하여 실제 결과의 확률 배열을 출력하세요.
3. `np.prod()`로 두 우도를 계산해 소수점 아래 세 자리로 출력하세요.
4. Q1~Q4에 답하세요.

#### 작성 도움말

두 번째 고객은 미구매입니다. 모델 A에서는 1−0.4를, 모델 B에서는 1−0.2를 사용해야 합니다. 구매 확률 배열을 곧바로 곱하지 않도록 주의하세요.

#### 해석 질문

**Q1.** 각 모델이 실제 결과에 부여한 확률 배열은 무엇인가요?

**Q2.** 두 모델의 우도와 전체 우도 기준으로 선택할 모델은 무엇인가요?

**Q3.** 미구매한 둘째 고객에게 더 높은 관측 확률을 주는 모델은 무엇이며, 전체 선택과 같나요?

**Q4.** 모델 A의 우도 0.288은 모델 A가 참일 확률이 28.8%라는 뜻인가요?

#### 제출 결과

- 요구사항의 계산 결과
- Q1~Q4 답변

In [ ]:
# 문제 3-2. 고객별 확률이 다른 두 모델의 우도 비교하기
# 1. 문제에 주어진 배열과 값을 작성하세요.
# 2. 요구사항 순서대로 계산하고 출력하세요.
# 3. 아래 답변란에 계산 결과의 뜻을 적으세요.

In [ ]:
y= np.array([1, 0, 1])
p_a= np.array([0.8, 0.4, 0.6])
p_b= np.array([0.6, 0.2, 0.5])

In [ ]:
def likelihood(p, y):
    observed_prob = (p ** y) * ((1 - p) ** (1 - y))
    return np.prod(observed_prob)

In [ ]:
print("likelihood(p_a, y): \n", likelihood(p_a, y))
print("\nlikelihood(p_b, y): \n", likelihood(p_b, y))

likelihood(p_a, y): 
 0.288

likelihood(p_b, y): 
 0.24


#### 답변 작성란

**Q1.** 각 모델이 실제 결과에 부여한 확률 배열은 무엇인가요?

- 답변: 모델 A는 [0.8, 0.6, 0.6], 모델 B는 [0.6, 0.8, 0.5]입니다.
- 구매한 고객(y=1)에게는 p를, 미구매한 둘째 고객(y=0)에게는 1−p를 사용했기 때문에 둘째 고객의 값이 각각 0.6(=1−0.4), 0.8(=1−0.2)이 됩니다.

**Q2.** 두 모델의 우도와 전체 우도 기준으로 선택할 모델은 무엇인가요?

- 답변: p_a 모델이 우도가 더 크므로, p_a를 선택해야 한다.

**Q3.** 미구매한 둘째 고객에게 더 높은 관측 확률을 주는 모델은 무엇이며, 전체 선택과 같나요?

- 답변: 둘째 고객에게는 모델 B가 0.8로, 모델 A의 0.6보다 높은 확률을 줍니다.
- 하지만 전체 우도는 모델 A가 더 크므로 전체 선택(모델 A)과는 다릅니다. 첫째·셋째 고객에서 모델 A가 더 높은 확률을 준 차이가 합쳐졌기 때문입니다.

**Q4.** 모델 A의 우도 0.288은 모델 A가 참일 확률이 28.8%라는 뜻인가요?

- 답변: 아닙니다. 0.288은 모델 A가 주어졌을 때 실제 관측 데이터가 나올 확률(모델 A가 관측 데이터에 부여한 확률)입니다.
- 우도는 모델이 참일 확률이 아니라, 관측 데이터를 얼마나 잘 설명하는지 모델끼리 비교하는 값입니다.

---

## 실습 마무리

다음 질문에 한두 문장으로 답하며 복습하세요.

**1.** 조건부 구매 비율의 분모는 누구인가요?

**2.** 실제 결과가 1 또는 0이면 각각 어떤 확률을 쓰나요?

**3.** Likelihood와 MLE는 어떻게 다른가요?

**4.** 공통 p를 사용하는 독립 베르누이 모델의 MLE는 어떻게 구하나요?

**5.** 전체 로그우도, Total NLL, Mean NLL의 관계는 무엇인가요?

**6.** 같은 데이터에서 우도·로그우도·NLL은 각각 어느 방향을 선택하나요?

### 실습 마무리 답변 작성란

**1.** 답변: 조건을 만족하는 고객 전체, 장바구니에 추가하겠다는 조건이면 추가한 고객을 분모로 한다.

**2.** 답변: 구매 혹률을 p라 할 때 y= 1에는 p, y= 0에는 (1 - p)를 사용
- 단, p는 항상 실제 결과의 확률인 것은 아님.

**3.** 답변: Likelihood는 한 파라미터에서 관측 데이터에 부여한 확률을 비교하는 값입니다.
- MLE는 그 우도를 가장 크게 만드는 파라미터를 찾는 방법

**4.** 답변: 구매 고객 수를 전체 고객 수로 나눈다. 0, 1, 실제 값 배열의 평균 np.mean(y)와 같음

**5.** 답변: 관측 확률의 자연로그를 모두 더하면 전체 로그우도, 그 부호를 바꾸면 Total NLL
- Total NLL을 데이터 개수로 나누면 Mean NLL이다.

**6.** 답변: 우도와 로그 우도는 큰 값, NLL은 작은 값을 선택한다.
- 양수에서 로그는 크기 순서를 유지하고, 마이너스를 붙이면 순서가 뒤집힘